# TRUEFRAME training and held-out evaluation (Google Colab T4)

Select **Runtime → Change runtime type → T4 GPU** before running. This notebook clones the repository and keeps data and run artifacts in Google Drive. You must obtain the datasets separately and follow their licenses. The project is a genuine-vs-AI-generated face research prototype, not a general deepfake detector.

In [ ]:
from pathlib import Path
import subprocess
repo_dir = Path('/content/trueframe')
if (repo_dir / '.git').exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/chsrikar/trueframe.git', str(repo_dir)], check=True)
%cd /content/trueframe
!pip install -q -r requirements-colab.txt

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import os, torch
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(0))

# Dataset files download to Colab temporary storage; only run artifacts are kept in Drive.
os.environ['TRUEFRAME_DATA_DIR'] = '/content/trueframe-data'
Path(os.environ['TRUEFRAME_DATA_DIR']).mkdir(parents=True, exist_ok=True)
run_dir = Path('/content/drive/MyDrive/trueframe-runs') / datetime.now().strftime('%Y%m%d-%H%M%S')
run_dir.mkdir(parents=True, exist_ok=True)
os.environ['TRUEFRAME_MANIFEST_DIR'] = str(run_dir / 'manifests')
os.environ['TRUEFRAME_CHECKPOINT_DIR'] = str(run_dir / 'checkpoints')
os.environ['TRUEFRAME_OUTPUT_DIR'] = str(run_dir / 'outputs')
for key in ('TRUEFRAME_MANIFEST_DIR', 'TRUEFRAME_CHECKPOINT_DIR', 'TRUEFRAME_OUTPUT_DIR'):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

## Download and prepare the datasets

This downloads about 24 GB of source archives/shards from Hugging Face and extracts a stratified 40,000-image synthetic sample to Colab temporary disk. Allow time and at least 32 GiB free disk. These files disappear when the runtime is deleted. The CelebA-HQ mirror currently lists its license as unknown; review the [dataset card](https://huggingface.co/datasets/saitsharipov/CelebA-HQ) before academic use or redistribution. The synthetic source is [SFHQ-T2I](https://github.com/SelfishGene/SFHQ-T2I-dataset).


In [ ]:
# Download public dataset files. No Kaggle API key is needed.
import subprocess, sys
subprocess.run([sys.executable, 'download_colab_data.py'], check=True)


In [ ]:
# Build generator-disjoint train/validation/test manifests. This stops if
# generator metadata is missing or fewer than three generator families exist.
import subprocess, sys
subprocess.run([sys.executable, 'build_manifest.py'], check=True)
print((Path(os.environ['TRUEFRAME_MANIFEST_DIR']) / 'split_protocol.json').read_text())

In [ ]:
# Train. Lower batch size to 16 if the selected GPU runs out of memory.
manifest_dir = Path(os.environ['TRUEFRAME_MANIFEST_DIR'])
subprocess.run([
    sys.executable, 'train.py',
    '--train-manifest', str(manifest_dir / 'manifest_train.csv'),
    '--val-manifest', str(manifest_dir / 'manifest_val.csv'),
    '--batch-size', '32', '--num-workers', '2', '--seed', '42',
    '--phase1-epochs', '3', '--phase2-epochs', '14'
], check=True)

In [ ]:
# Evaluate once after training. The report contains overall and per-source /
# per-generator metrics plus calibration diagnostics. Do not tune on this test set.
subprocess.run([sys.executable, 'evaluate.py'], check=True)
print('Saved reports to:', os.environ['TRUEFRAME_OUTPUT_DIR'])
print('Saved model to:', os.environ['TRUEFRAME_CHECKPOINT_DIR'])